# Runs

Reads the main runs and the pilot's phase 2, which starts as they do, pooled wherever both exist, at H ([`README.md`](README.md) § Hypotheses and tests, § The fit;
[`schedule.md`](schedule.md) § Pilot, § Runs): every run's outcome at round H + 1, its
declarations and turns up to it. Per cell and condition: the outcomes, the declarations'
signature table, the fits, the declaration tests and the validity
check, the outcome tests and H3, the self-reads, the specification tables. `STAGES` holds one
directory per stage, each run `run_<sub-stage>_r1` as `run_stage.sh` names it, copied back from
the SCC; until the main runs exist it reads the pilot alone.

## 1. Load

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.precision", 3)  # every table at three decimals

CWD = Path.cwd().resolve()
REPO = next(p for p in [CWD, *CWD.parents] if (p / "simulation" / "config.py").is_file())
sys.path.insert(0, str(REPO / "analysis"))
import fit  # noqa: E402
import metrics  # noqa: E402
import config  # noqa: E402  (fit puts simulation/ on the path)

import recovery  # noqa: E402
import robustness  # noqa: E402
import surrogate  # noqa: E402
from surrogate import Pool  # noqa: E402


def r3(x):
    """x with every float rounded to three decimals, through dicts, lists and tuples."""
    if isinstance(x, dict):
        return {k: r3(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return type(x)(r3(v) for v in x)
    return round(float(x), 3) if isinstance(x, float) else x


LOGS = REPO / "analysis" / "logs"
STAGES = {"P2": LOGS / "pilot" / "phase2", "M": LOGS / "runs"}   # pooled wherever both exist (schedule.md § Pilot)
AUDIT = LOGS / "audit"
N_REP = None     # the outcome tests wait for n_rep (schedule.md § Runs)

found = sorted(d for stage, root in STAGES.items() if root.is_dir()
               for d in root.glob(f"run_{stage}-*") if (d / "params.json").is_file())
dirs = [d for d in found if metrics.is_complete(d)]  # runs still going or aborted wait
H = config.NETWORK_MAX_ROUNDS   # fixed by design (schedule.md § Runs, Horizon)
off = [d.name for d in dirs if metrics.read_params(d)["NETWORK_MAX_ROUNDS"] != H]
assert not off, f"runs not at H = {H}: {off}"
groups: dict = {}
for d in dirs:
    p = metrics.read_params(d)
    groups.setdefault((p["TOPIC_KEY"], p["LLM_MODEL"], p["CONDITION"]), []).append(d)
stages = pd.Series([d.name.split("_")[1].split("-")[0] for d in dirs]).value_counts().astype(int)
print(len(dirs), "complete runs in", len(groups), f"groups (by stage: {stages.to_dict()});",
      len(found) - len(dirs), "unfinished skipped; H =", H)

## 2. Outcomes and H3

Each run's divergence D at round H + 1, its final declarations, and H3's permutation
test on D's slope on p_inter with the isotonic transition, beside the benchmarks' D̄(p_inter, H);
in `full`, also pooled per LLM, permuting within topic, outside the error control (README § Scope).

In [ ]:
outcomes = {g: pd.DataFrame([metrics.run_outcome(d, at=H + 1) for d in ds]) for g, ds in groups.items()}
for g, df in outcomes.items():
    slope, p = fit.coupling_test(df.p_inter, df.divergence)
    print(g, f"mean D {df.divergence.mean():.3f}, slope {slope:.3f}, p {p:.3f}, "
             f"transition {fit.transition(df.p_inter, df.divergence):.3f}")
for llm in sorted({m for _, m, c in outcomes if c == "full"}):
    df = pd.concat([d.assign(topic=t) for (t, m, c), d in outcomes.items() if m == llm and c == "full"])
    slope, p = fit.coupling_test(df.p_inter, df.divergence, df.topic)
    print(llm, f"pooled: mean D {df.divergence.mean():.3f}, slope {slope:.3f}, p {p:.3f}")

## 3. Declarations: the signature table

The signature table (README § The models, Signatures), on all entries, on explored ones and on the rest: an
explored entry's own pole is drawn, so explored entries crowd the cells where ô
disagrees with ŝ and lower their y through the return to the own pole, `own` included.

In [ ]:
decls = {g: [x for d in ds for x in metrics.run_declarations(d) if x["round"] <= H + 1]
         for g, ds in groups.items()}
data = {g: fit.arrays(v) for g, v in decls.items() if v}
for g, dt in data.items():
    print(g, "all entries, then explored ones, then the rest")
    display(metrics.signatures(decls[g]))
    for explored in (True, False):
        display(metrics.signatures([d for d in decls[g]
                                    if d["window"] and d["explored"][-1] == explored]))

## 4. The fit

M0–M3 pooled per group: parameters, BIC (descriptive) and fits at a bound.

In [ ]:
fits = {g: {m: fit.fit(dt, m) for m in fit.MODELS} for g, dt in data.items()}
for g, fs in fits.items():
    print(g)
    display(pd.DataFrame({m: {"ll": f["ll"], "bic": f["bic"], "at_bound": f["at_bound"], **f["params"]}
                          for m, f in fs.items()}))

## 5. The declaration tests, the validity check and the outcome tests

Held-out gains per run (`recovery.declarations`), H1, H2 and H3 at their levels
(`recovery.decide`); in `own`, the validity check against `full`; the outcome test's CRPS from the
surrogate on each run's graph and starts, the exchanges replayed from the cell's audit; M1's and
M2's gains over M0, descriptive.

In [ ]:
results, run_sets, outcome_args = {}, {}, {}
for (topic, model, cond), ds in groups.items():
    g = (topic, model, cond)
    if g not in data:
        continue
    runs = []
    for d, row in zip(ds, outcomes[g].itertuples()):
        graph, starts = surrogate.run_setup(d)
        runs.append({"p_inter": row.p_inter, "divergence": row.divergence, "seed": d.name, "graph": graph,
                     "starts": starts, "values": row.values,
                     "divergence_half": metrics.run_outcome(d, at=H // 2)["divergence"]})
    cell = AUDIT / f"{topic}_{model.replace(':', '-')}"
    outcome_args[g] = ({"pool": Pool.from_audit(cell), "rounds": H, "n_rep": N_REP}
                       if cond == "full" and N_REP and (cell / "audit_meetings").is_dir() else None)
    run_sets[g] = {"runs": runs, "decls": decls[g]}
    results[g] = recovery.analyse(run_sets[g], outcome_args=outcome_args[g])
    print(g, r3(results[g]["p"]), results[g]["decisions"])
    print("  M1:M0, M2:M0, descriptive:", r3([float(results[g]["stats"][h].mean()) for h in ("M1:M0", "M2:M0")]))

for (topic, model, cond), res in results.items():
    full = results.get((topic, model, "full"))
    if cond == "own" and full:
        print((topic, model), "validity p", r3(fit.validity_test(res["stats"]["H1"], full["stats"]["H1"])))

## 6. Self-reads

Per run: the share of turns whose self-read ŝ differs from the
argued position (`inconsistent`), by argued position and exploration; the effective
exploration rate, the share of turns that are explored entries (`explored_entry`:
explored and read back on the pole argued) and the realised rate, the share whose
ŝ leaves the declared pole (`off_pole`), both against ε; and the channel's lean toward a
pole (`metrics.expression_bias`), which tilts consensus direction as answer order does.

In [ ]:
for g, ds in groups.items():
    t = pd.concat([metrics.run_turns(d).assign(run=d.name) for d in ds])
    t = t[t["round"] <= H]
    cols = ["inconsistent", "explored_entry", "off_pole"]
    t[cols] = t[cols].astype(bool)
    eps = metrics.read_params(ds[0])["EXPLORATION_RATE"]
    print(g, f"inconsistent turns {t.inconsistent.mean():.3f}, "
             f"explored entries {t.explored_entry.mean():.3f}, off pole {t.off_pole.mean():.3f} at ε = {eps}")
    display(t.groupby("run")[cols].mean().describe())
    display(t.pivot_table("inconsistent", "s", "explored", aggfunc="mean"))
    display(pd.DataFrame({r: metrics.expression_bias(tr) for r, tr in t.groupby("run")}).T)

## 7. Figures

Each shows runs or replicates whole beside any summary (Fogarty et al. 2022, Rule 10):
every run's gain per contrast; every run's D at its p_inter
with H3's isotonic fit beside both benchmarks' D̄(p_inter, H); every run's two community means of the
final positions by p_inter, and the final positions' distribution; each cell's ladder
(README § The references, The ladder), by p_inter at H, the first three also at 20,000 and the fitted rules' also at 2H.

## 8. Specification tables

What each test's conclusion rests on ([`schedule.md`](schedule.md) § Sensitivity): every
variant's result and whether it agrees with the primary reading. Uncorrected, outside the
error control, gating nothing; a flip is reported as a limit of the conclusion.

In [ ]:
for g, df in outcomes.items():
    display(g, robustness.coupling_table(df))
for g in results:
    if g[2] == "full":
        display(g, robustness.declarations_table(decls[g]))
    if outcome_args[g]:
        other_runs = Pool.from_other_runs(groups[g], outcome_args[g]["pool"])
        display(g, robustness.outcomes_table(run_sets[g], results[g]["stats"]["fits"], other_runs=other_runs,
                                             **outcome_args[g]))